# Process model learning results

In [ ]:
import numpy as np
import h5py
from matplotlib import pyplot as plt
from matplotlib import patches as patches
from matplotlib import ticker as ticker
from os import listdir
from os.path import isfile, join
from scipy.stats import gaussian_kde

Get the data files in alphabetical order:

In [ ]:
path  = "/home/giovanni/Documents/work/programming/matlab/data-driven-methods/stable-discovery-clean/ex02-dysts/models-basic/"
file_list = [f for f in listdir(path) if isfile(join(path, f))]
file_list.sort()
print("Number of files: ", len(file_list))

## Create lists of polynomial and other models

First, we list all polynomial models.

In [ ]:
poly_ode_names = [
    "Aizawa",
    "Arneodo",
    "AtmosphericRegime",
    "Bouali",
    "Bouali2",
    "BurkeShaw",
    "Chen",
    "ChenLee",
    "Coullet",
    "Dadras",
    "DequanLi",
    "Finance",
    "GenesioTesi",
    "GuckenheimerHolmes",
    "Hadley",
    "Halvorsen",
    "HenonHeiles",
    "HindmarshRose",
    "HyperBao",
    "HyperCai",
    "HyperJha",
    "HyperLorenz",
    "HyperLu",
    "HyperPang",
    "HyperQi",
    "HyperRossler",
    "HyperWang",
    "HyperXu",
    "HyperYan",
    "HyperYangChen",
    "IsothermalChemical",
    "KawczynskiStrizhak",
    "Laser",
    "LiuChen",
    "Lorenz",
    "Lorenz84",
    "Lorenz96",
    "LorenzBounded",
    "LorenzCoupled",
    "LorenzStenflo",
    "LuChen",
    "LuChenCheng",
    "MooreSpiegel",
    "NewtonLiepnik",
    "NoseHoover",
    "NuclearQuadrupole",
    "PanXuZhou",
    "PehlivanWei",
    "Qi",
    "QiChen",
    "RabinovichFabrikant",
    "RayleighBenard",
    "RikitakeDynamo",
    "Rossler",
    "Rucklidge",
    "Sakarya",
    "ShimizuMorioka",
    "SprottA",
    "SprottB",
    "SprottC",
    "SprottD",
    "SprottE",
    "SprottF",
    "SprottG",
    "SprottH",
    "SprottI",
    "SprottJ",
    "SprottJerk",
    "SprottK",
    "SprottL",
    "SprottM",
    "SprottN",
    "SprottO",
    "SprottP",
    "SprottQ",
    "SprottR",
    "SprottS",
    "SprottTorus",
    "Tsucs2",
    "VallisElNino",
    "WangSun",
    "ZhouChen"]

Then, we list all other models:

In [ ]:
other_ode_names = [
    "AnishchenkoAstakhov",
    "BeerRNN",
    "BelousovZhabotinsky",
    "Blasius",
    "CaTwoPlus",
    "CaTwoPlusQuasiperiodic",
    "CellCycle",
    "CellularNeuralNetwork",
    "Chua",
    "CoevolvingPredatorPrey",
    "Colpitts",
    "ExcitableCell",
    "GlycolyticOscillation",
    "HastingsPowell",
    "Hopfield",
    "ItikBanksTumor",
    "JerkCircuit",
    "MacArthur",
    "MultiChua",
    "SaltonSea",
    "SanUmSrisuchinwong",
    "SprottMore",
    "Thomas",
    "ThomasLabyrinth",
    "Torus",
    "WindmiReduced",
    "YuWang",
    "YuWang2"]

Finally, we sort things out a little:

In [ ]:
poly_odes = []
poly_ode_files = []
other_odes = []
other_ode_files = []

for name in poly_ode_names:
    for s in file_list:
        if ("_" + name + ".hdf5") in s:
            poly_odes.append(name)
            poly_ode_files.append(s)

for name in other_ode_names:
    for s in file_list:
        if ("_" + name + ".hdf5") in s:
            other_odes.append(name)
            other_ode_files.append(s)

print("Polynomial ODEs: ", len(poly_ode_files))
print("Other ODEs: ", len(other_ode_files))

## Make violin plots -- horizontal layout

In [ ]:
# General plot parameters
fig_width  = 15
sys_height = 0.425
colors =  [ '#f1c40f', '#9b59b6', '#e67e22', '#ff5733', '#3498db', '#2ecc71']
violin_shift = 0.0
nCols = 2
wspace = 0.75
font_size = 9

In [ ]:
## Plot for the polynomial models
shift = int( np.ceil( len(poly_odes) / nCols ) )
fig_height = shift * sys_height
print("Figure width  =", fig_width)
print("Figure height =", fig_height)
fig, ax_list = plt.subplots(1, nCols, figsize=(fig_width/2.54, fig_height/2.54))
ax_counter = 0
inaccurate = 0
total = 0
all_degs = []

for ax in ax_list:
    counter = 0
    systems_to_plot = poly_ode_files[ax_counter*shift:(ax_counter+1)*shift]
    system_names = poly_odes[ax_counter*shift:(ax_counter+1)*shift]
    ax_counter += 1
    nFiles = len(systems_to_plot)
    print(nFiles)
    ax.axvspan(0, 1e-2, facecolor='green', alpha=0.1, edgecolor="none")
    
    for system in systems_to_plot:
        
        # load error data
        total += 1
        counter += 1
        filename = path + system
        fobj = h5py.File(filename, "r")
        err_raw = fobj["model/error/raw"][()].squeeze()
        err_avg = fobj["model/error/avg"][()].squeeze()
        model_deg = fobj["model/degree"][()].squeeze() # starts at 2!
        fobj.close()
        
        # Count inaccurate models
        if err_avg > 1e-2: inaccurate += 1
        
        # Accumulate degrees
        all_degs.append(model_deg)

        # select color based on model degree
        clr = colors[int(model_deg)-2]
        
        # Make violin plot
        position = nFiles-counter+violin_shift
        violin = ax.violinplot(err_raw, 
                    positions=[position], orientation='horizontal', 
                    widths=1-violin_shift, showmeans=False, showextrema=False, showmedians=False, quantiles=None,
                    points=500, bw_method=None, side='high', data=None)
        violin['bodies'][0].set_facecolor(clr)#.squeeze()
        
        # Plot mean
        ax.plot([err_avg, err_avg], [position+0.05, position+0.9], linewidth=1.5, color=clr)
        
        # Plot the density line
        x = violin['bodies'][0].get_paths()[0].vertices[:, 0]
        y = violin['bodies'][0].get_paths()[0].vertices[:, 1]
        xup = [x[k] for k in range(len(x)) if y[k] > position]
        yup = [y[k] for k in range(len(y)) if y[k] > position]
        ax.plot(xup[1:], yup[1:], color=clr, linewidth=0.75, alpha=1) 
    
    # Format plot
    major_ticks = [n-1 for n in range(nFiles, 0, -1)]
    minor_ticks = [n-0.5 for n in range(nFiles, 0, -1)]
    ax.set_yticks(minor_ticks, minor=True)
    ax.set_yticks(major_ticks, minor=False)
    ax.set_yticklabels(system_names, minor=True, size=font_size)
    ax.set_yticklabels([], minor=False)
    ax.tick_params(axis='y', which='minor', length=0)
    ax.set_xscale('log')
    ax.set_xticks([10**n for n in range(-10,4,2)], minor=False)
    ax.set_xticklabels([f"$10^{{{n}}}$" for n in range(-10,4,2)],size=font_size)
    ax.set_xlabel("Model Error", size=10)
    ax.set_xlim([1e-10,1e2])
    ax.set_ylim([nFiles-shift,nFiles])
    ax.grid(True, which='major', axis='y', linestyle='-', linewidth=0.25)
    ax.grid(True, which='major', axis='x', linestyle='-', linewidth=0.25)

# Tight layout
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=wspace)
plt.savefig('dtsys_poly.pdf', dpi=1200, bbox_inches='tight')

# Diagnostics
print(" Total num models:", total)
print("Inaccurate models:", inaccurate)

In [ ]:
# Plot for the non-polynomial models (expect bad results)
shift = int( np.ceil( len(other_odes) / nCols ) )
fig_height = shift * sys_height
print("Figure width  =", fig_width)
print("Figure height =", fig_height)
fig, ax_list = plt.subplots(1, nCols, figsize=(fig_width/2.54, fig_height/2.54))
ax_counter = 0
inaccurate = 0
total = 0
all_degs = []

for ax in ax_list:
    counter = 0
    systems_to_plot = other_ode_files[ax_counter*shift:(ax_counter+1)*shift]
    system_names = other_odes[ax_counter*shift:(ax_counter+1)*shift]
    ax_counter += 1
    nFiles = len(systems_to_plot)
    print(nFiles)
    ax.axvspan(0, 1e-2, facecolor='green', alpha=0.1, edgecolor="none")
    
    for system in systems_to_plot:
        
        # load error data
        total += 1
        counter += 1
        filename = path + system
        fobj = h5py.File(filename, "r")
        err_raw = fobj["model/error/raw"][()].squeeze()
        err_avg = fobj["model/error/avg"][()].squeeze()
        model_deg = fobj["model/degree"][()].squeeze() # starts at 2!
        fobj.close()
        
        # Count inaccurate models
        if err_avg > 1e-2: inaccurate += 1
        
        # Accumulate degrees
        all_degs.append(model_deg)

        # select color based on model degree
        clr = colors[int(model_deg)-2]
        
        # Make violin plot
        position = nFiles-counter+violin_shift
        violin = ax.violinplot(err_raw, 
                    positions=[position], orientation='horizontal', 
                    widths=1-violin_shift, showmeans=False, showextrema=False, showmedians=False, quantiles=None,
                    points=500, bw_method=None, side='high', data=None)
        violin['bodies'][0].set_facecolor(clr)#.squeeze()
        
        # Plot mean
        ax.plot([err_avg, err_avg], [position+0.05, position+0.9], linewidth=1.5, color=clr)
        
        # Plot the density line
        x = violin['bodies'][0].get_paths()[0].vertices[:, 0]
        y = violin['bodies'][0].get_paths()[0].vertices[:, 1]
        xup = [x[k] for k in range(len(x)) if y[k] > position]
        yup = [y[k] for k in range(len(y)) if y[k] > position]
        ax.plot(xup[1:], yup[1:], color=clr, linewidth=0.75, alpha=1) 
    
    # Format plot
    major_ticks = [n-1 for n in range(nFiles, 0, -1)]
    minor_ticks = [n-0.5 for n in range(nFiles, 0, -1)]
    ax.set_yticks(minor_ticks, minor=True)
    ax.set_yticks(major_ticks, minor=False)
    ax.set_yticklabels(system_names, minor=True, size=font_size)
    ax.set_yticklabels([], minor=False)
    ax.tick_params(axis='y', which='minor', length=0)
    ax.set_xscale('log')
    ax.set_xticks([10**n for n in range(-6,4,2)], minor=False)
    ax.set_xticklabels([f"$10^{{{n}}}$" for n in range(-6,4,2)],size=font_size)
    ax.set_xlabel("Model Error", size=10)
    ax.set_xlim([1e-6,1e2])
    ax.set_ylim([nFiles-shift,nFiles])
    ax.grid(True, which='major', axis='y', linestyle='-', linewidth=0.25)
    ax.grid(True, which='major', axis='x', linestyle='-', linewidth=0.25)

# Tight layout
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=wspace)
plt.savefig('dtsys_other.pdf', dpi=1200, bbox_inches='tight')

# Diagnostics
print(" Total num models:", total)
print("Inaccurate models:", inaccurate)